In [1]:
import pandas as pd
import requests as rq
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import random
import time

In [2]:

def page_generator(page: int = 1) -> str:
    url_to_scrap = fr"""
    https://eur-lex.europa.eu/search.html?dom=LEGISLATION%2CTREATIES%2CEU_CASE_LAW%2CCONSLEG&SUBDOM_INIT=
    ALL_ALL&orDC_DOM_CODED=DC_TT_CODED%3D434743%2CDC_TT_CODED%3D2090%2CDC_TT_CODED%3D6011%2CDC_
    TT_CODED%3D2470%2CDC_TT_CODED%3D371%2CDC_TT_CODED%3D3150%2CDC_TT_CODED%3D3549%2CDC_TT_CODED%3D2524%2CDC_
    TT_CODED%3D343%2CDC_TT_CODED%3D1158%2CDC_TT_CODED%3Dc_98d1408a%2CDC_TT_CODED%3D59447%2CDC_TT_CODED%3D3146%2CDC_
    TT_CODED%3D833%2CDC_TT_CODED%3D1707%2CDC_TT_CODED%3D2825&lang=en&type=advanced&qid=1761853731500&wh0=
    andCOMPOSE%3DENG%2CorEMBEDDED_MANIFESTATION-TYPE%3Dpdf%3BEMBEDDED_MANIFESTATION-TYPE%3Dpdfa1a%3BEMBEDDED_MANIFESTATION-
    TYPE%3Dpdfa1b%3BEMBEDDED_MANIFESTATION-TYPE%3Dpdfa2a%3BEMBEDDED_MANIFESTATION-TYPE%3Dpdfx%3BEMBEDDED_MANIFESTATION-
    TYPE%3Dpdf1x%3BEMBEDDED_MANIFESTATION-TYPE%3Dhtml%3BEMBEDDED_MANIFESTATION-TYPE%3Dxhtml%3BEMBEDDED_MANIFESTATION-
    TYPE%3Ddoc%3BEMBEDDED_MANIFESTATION-TYPE%3Ddocx&page={page}
    """.replace("\n", "").replace(" ", "")
    return url_to_scrap


def get_number_of_pages() -> int:
    url = page_generator()
    # Options du navigateur (sans interface graphique)
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # supprime cette ligne pour voir le navigateur
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "label"))
        )
    except:
        driver.quit()
        return

    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, "html.parser")

    strongs = soup.find_all("strong")
    nbr_page = int(int(strongs[2].get_text(strip=True))/10)
    return nbr_page

In [3]:
def get_data_from_div(soup: BeautifulSoup) -> list[dict]:
    data_list = []
    divs = soup.find_all("div", attrs={"xmlns": "http://www.w3.org/1999/xhtml"})

    for div in divs:
        entry = {}

        # Titre
        h2 = div.find("h2")
        entry["title"] = h2.get_text(strip=True) if h2 else None

        # Paragraphes
        p_tags = div.find_all("p")
        entry["ref"] = p_tags[0].get_text(strip=True) if len(p_tags) > 0 else None
        entry["details"] = p_tags[1].get_text(strip=True) if len(p_tags) > 1 else None

        # Statut
        status_p = div.find("p", class_="forceindicator")
        entry["status"] = status_p.get_text(strip=True) if status_p else None

        # paires <dt>/<dd>
        for dt, dd in zip(div.find_all("dt"), div.find_all("dd")):
            entry[dt.get_text(strip=True)] = dd.get_text(strip=True)

        # Lien
        a_tag = div.find("a", href=True)
        entry["link"] = a_tag["href"] if a_tag else None

        data_list.append(entry)

    return data_list


In [4]:
# data = get_data_from_div()
# for i, d in enumerate(data, 1):
#     print(f"\n--- Résultat {i} ---")
#     for k, v in d.items():
#         print(f"{k} {v}")

In [ ]:
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

CHECKPOINT_FILE = "checkpoint_data.csv"

def get_data_from_page() -> pd.DataFrame:
    all_data = []
    nbr_page = get_number_of_pages() + 1

    # Reprise depuis un checkpoint si existant
    if os.path.exists(CHECKPOINT_FILE):
        df_checkpoint = pd.read_csv(CHECKPOINT_FILE)
        all_data = df_checkpoint.to_dict("records")
        start_page = df_checkpoint["page"].max() + 1
        print(f"Reprise à la page {start_page}/{nbr_page}")
    else:
        start_page = 1

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=chrome_options)

    for current_page in range(start_page, nbr_page + 1):
        url = page_generator(current_page)
        print(f"[{current_page}/{nbr_page}] → {url}")
        driver.get(url)

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "label"))
            )
        except:
            print(f"Timeout page {current_page}, passage à la suivante.")
            continue

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        data = get_data_from_div(soup)

        if data:
            # Ajouter un champ de suivi de la page
            for d in data:
                d["page"] = current_page
            all_data.extend(data)

            # Sauvegarde temporaire après chaque page
            pd.DataFrame(all_data).to_csv(CHECKPOINT_FILE, index=False)
            print(f"✔ Page {current_page} sauvegardée ({len(all_data)} entrées).")

        # Délai aléatoire anti-blocage
        delay = random.uniform(3, 8)
        print(f"Pause {delay:.2f}s avant la page suivante.")
        time.sleep(delay)

    driver.quit()

    # Sauvegarde finale consolidée
    df = pd.DataFrame(all_data)
    df.to_csv("list_of_items.csv", index=False)
    print(f"✅ Scraping terminé. Total {len(df)} lignes.")
    return df


In [13]:
df.shape

(9380, 13)

In [17]:
df = get_data_from_page()

Reprise à la page 936/939


KeyboardInterrupt: 